<a href="https://colab.research.google.com/github/baramitha/bt4222/blob/main/Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Loading


In [ ]:
import pandas as pd

anime_clean = pd.read_csv('/content/anime_clean.csv')
users_sampled = pd.read_csv('/content/users_sampled_engineered.csv')
ratings_sampled = pd.read_csv('/content/ratings_sampled_engineered.csv')

# **Data Preparation**

In [17]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

In [20]:
model_data = ratings_sampled[['user_id', 'anime_id', 'rating']].copy()
model_data = model_data.rename(columns={'anime_id': 'movie_id'})
model_data = model_data[model_data['rating'] >0].reset_index(drop=True)
# 20% subsample for baseline
model_data = model_data.sample(frac=0.2, random_state=42).reset_index(drop=True)

# Re-encode IDs
user_encoder= LabelEncoder()
movie_encoder = LabelEncoder()
model_data['user_id']  = user_encoder.fit_transform(model_data['user_id'])
model_data['movie_id'] = movie_encoder.fit_transform(model_data['movie_id'])

# Split: 80% train, 10% val, 10% test
train_data, temp_data = train_test_split(model_data, test_size=0.2, random_state=42)
val_data, test_data= train_test_split(temp_data,  test_size=0.5, random_state=42)

train_data = train_data.reset_index(drop=True)
val_data= val_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

# Filter val and test to seen users/items (from train only)
train_users = set(train_data['user_id'].unique())
train_items = set(train_data['movie_id'].unique())

filtered_val_data = val_data[
    val_data['user_id'].isin(train_users) & val_data['movie_id'].isin(train_items)
].reset_index(drop=True)

filtered_test_data = test_data[
    test_data['user_id'].isin(train_users) & test_data['movie_id'].isin(train_items)
].reset_index(drop=True)
num_users  = len(user_encoder.classes_)
num_movies = len(movie_encoder.classes_)
print(f'Train: {len(train_data):,} | Val: {len(filtered_val_data):,} | Test: {len(filtered_test_data):,}')
print(f'Users: {num_users:,} | Anime: {num_movies:,}')
print(f'Rating range: {model_data["rating"].min():.0f}–{model_data["rating"].max():.0f}')

Train: 3,677,446 | Val: 458,037 | Test: 458,015
Users: 163,930 | Anime: 15,061
Rating range: 1–10


# **Matrix factorisation**

In [21]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
# Dataset
class AnimeDataset(Dataset):
    def __init__(self, data):
        data = data.reset_index(drop=True)
        self.user_ids  = torch.tensor(data['user_id'].values,dtype=torch.long).to(device)
        self.movie_ids = torch.tensor(data['movie_id'].values, dtype=torch.long).to(device)
        self.ratings   = torch.tensor(data['rating'].values,dtype=torch.float).to(device)
    def __len__(self):
        return len(self.ratings)
    def __getitem__(self, idx):
        return {
            'user_id':self.user_ids[idx],'movie_id': self.movie_ids[idx],'rating': self.ratings[idx]
        }
batch_size = 2048
train_loader = DataLoader(AnimeDataset(train_data),batch_size=batch_size, shuffle=True,  num_workers=0, pin_memory=False)
val_loader= DataLoader(AnimeDataset(filtered_val_data),batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)
test_loader = DataLoader(AnimeDataset(filtered_test_data), batch_size=batch_size, shuffle=False, num_workers=0, pin_memory=False)
print(f'Train batches: {len(train_loader)} |Test batches: {len(test_loader)}')


Using device: cuda
Train batches: 1796 |Test batches: 224


In [22]:
class MatrixFactorization(nn.Module):
    def __init__(self, num_users, num_movies, embedding_dim):
        super(MatrixFactorization, self).__init__()
        self.user_embeddings = nn.Embedding(num_users, embedding_dim)
        self.item_embeddings = nn.Embedding(num_movies, embedding_dim)
    def forward(self, user_id, movie_id):
        user_emb = self.user_embeddings(user_id)
        item_emb = self.item_embeddings(movie_id)
        return torch.clamp((user_emb * item_emb).sum(dim=-1), 1, 10)
embedding_dim = 16
model     = MatrixFactorization(num_users, num_movies, embedding_dim).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(f'Model device: {next(model.parameters()).device}')
#Training
best_loss = float('inf')
patience = 2
patience_counter = 0
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        user_id  = batch['user_id']
        movie_id = batch['movie_id']
        ratings= batch['rating']
        optimizer.zero_grad()
        loss = criterion(model(user_id, movie_id), ratings)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_train_loss = epoch_loss /len(train_loader)
    print(f'Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f}', end=' | ')

    model.eval()
    eval_loss = 0
    with torch.no_grad():
        for batch in val_loader:
            user_id  = batch['user_id']
            movie_id = batch['movie_id']
            ratings  = batch['rating']
            eval_loss += criterion(model(user_id, movie_id), ratings).item()
    avg_eval_loss = eval_loss/len(val_loader)
    print(f'Val Loss: {avg_eval_loss:.4f}')
    if avg_eval_loss<best_loss:
        best_loss= avg_eval_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_model.pt')
        print(f'Best model saved (loss: {best_loss:.4f})')
    else:
        patience_counter += 1
        print(f'No improvement. Patience: {patience_counter}/{patience}')
        if patience_counter >= patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

Model device: cuda:0
Epoch 1/10 | Train Loss: 35.4698 | Val Loss: 34.7166
Best model saved (loss: 34.7166)
Epoch 2/10 | Train Loss: 33.6341 | Val Loss: 33.7333
Best model saved (loss: 33.7333)
Epoch 3/10 | Train Loss: 32.1019 | Val Loss: 33.0147
Best model saved (loss: 33.0147)
Epoch 4/10 | Train Loss: 30.8342 | Val Loss: 32.4650
Best model saved (loss: 32.4650)
Epoch 5/10 | Train Loss: 29.7798 | Val Loss: 32.0344
Best model saved (loss: 32.0344)
Epoch 6/10 | Train Loss: 28.8980 | Val Loss: 31.6906
Best model saved (loss: 31.6906)
Epoch 7/10 | Train Loss: 28.1547 | Val Loss: 31.4114
Best model saved (loss: 31.4114)
Epoch 8/10 | Train Loss: 27.5215 | Val Loss: 31.1793
Best model saved (loss: 31.1793)
Epoch 9/10 | Train Loss: 26.9761 | Val Loss: 30.9832
Best model saved (loss: 30.9832)
Epoch 10/10 | Train Loss: 26.4997 | Val Loss: 30.8158
Best model saved (loss: 30.8158)


In [23]:
torch.save(model.state_dict(), 'best_model.pt')
print('Saved!')

Saved!


In [24]:
# 6. eval
print('=' * 50)
model.load_state_dict(torch.load('best_model.pt', map_location=device))
model.eval()
all_predictions, all_ratings_list = [], []
with torch.no_grad():
    for batch in test_loader:
        user_id  = batch['user_id']
        movie_id = batch['movie_id']
        ratings  = batch['rating']
        outputs  = model(user_id, movie_id)
        all_predictions.extend(outputs.cpu().numpy())
        all_ratings_list.extend(ratings.cpu().numpy())
all_predictions = np.array(all_predictions)
all_ratings_arr = np.array(all_ratings_list)
rmse = np.sqrt(np.mean((all_predictions - all_ratings_arr) ** 2))
mae = np.mean(np.abs(all_predictions - all_ratings_arr))
print(f'Final RMSE: {rmse:.4f}')
print(f'Final MAE:  {mae:.4f}')

Final RMSE: 5.5583
Final MAE:  4.9059


In [25]:
model = MatrixFactorization(num_users, num_movies, embedding_dim).to(device)
model.load_state_dict(torch.load('best_model.pt', map_location=device))
model.eval()

MatrixFactorization(
  (user_embeddings): Embedding(163930, 16)
  (item_embeddings): Embedding(15061, 16)
)

In [26]:
def generate_predictions(model, data_loader):
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in data_loader:
            user_id  = batch['user_id']
            movie_id = batch['movie_id']
            outputs  = model(user_id, movie_id)
            predictions.extend(zip(user_id.cpu().numpy(), movie_id.cpu().numpy(), outputs.cpu().numpy()))
    return predictions
test_predictions = generate_predictions(model, test_loader)
mf_predictions = test_predictions

# Convert test data to DataFrame for easier manipulation
test_data_df = pd.DataFrame(filtered_test_data, columns=['user_id', 'movie_id', 'rating'])
# Map actual ratings to relevance scores
test_data_df['relevance'] = test_data_df['rating'].apply(lambda x: x if x > 0 else 0)  # Example relevance mapping
test_data_df
len(test_data_df)
len(test_predictions)

458015

In [27]:
from collections import defaultdict
def calculate_rmse(predictions, actuals):
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    return np.sqrt(np.mean((predictions - actuals) ** 2))

def calculate_mae(predictions, actuals):
    predictions = np.array(predictions)
    actuals = np.array(actuals)
    return np.mean(np.abs(predictions - actuals))

# RANKING METRICS
def dcg_at_k(relevance_scores, k):
    relevance_scores = np.asarray(relevance_scores)[:k]
    if relevance_scores.size:
        return np.sum((2**relevance_scores - 1) / np.log2(np.arange(2, relevance_scores.size + 2)))
    return 0.0

def ndcg_at_k(relevance_scores, ideal_relevance_scores, k):

    dcg = dcg_at_k(relevance_scores, k)
    idcg = dcg_at_k(ideal_relevance_scores, k)
    if idcg == 0:
        return 0.0
    return dcg/idcg


def precision_at_k(recommended_items, relevant_items, k):
    recommended_at_k = set(recommended_items[:k])
    relevant_set = set(relevant_items)
    hits = len(recommended_at_k & relevant_set)
    return hits /k if k > 0 else 0.0

def recall_at_k(recommended_items, relevant_items, k):
    recommended_at_k = set(recommended_items[:k])
    relevant_set = set(relevant_items)
    hits = len(recommended_at_k & relevant_set)
    return hits /len(relevant_set) if len(relevant_set) > 0 else 0.0

def hit_rate_at_k(recommended_items, relevant_items, k):
    recommended_at_k = set(recommended_items[:k])
    relevant_set = set(relevant_items)
    return 1.0 if len(recommended_at_k & relevant_set) > 0 else 0.0

def reciprocal_rank(recommended_items, relevant_items):
    relevant_set = set(relevant_items)
    for rank, item in enumerate(recommended_items, 1):
        if item in relevant_set:
            return 1.0 / rank
    return 0.0

def average_precision(recommended_items, relevant_items):
    relevant_set = set(relevant_items)
    hits = 0
    sum_precision = 0.0
    for rank, item in enumerate(recommended_items, 1):
        if item in relevant_set:
            hits += 1
            sum_precision += hits / rank

    return sum_precision / len(relevant_set) if len(relevant_set) > 0 else 0.0

def catalog_coverage(all_recommendations, total_items):
    unique_recommended = set(all_recommendations)
    return len(unique_recommended) / total_items if total_items > 0 else 0.0


def user_coverage(users_with_recommendations, total_users):
    return users_with_recommendations / total_users if total_users > 0 else 0.0

class RecommenderMetrics:
    def __init__(self, relevance_threshold=4):
        self.relevance_threshold = relevance_threshold

    def evaluate_all(self, predictions_df, test_data_df, k_values=[5, 10, 20],
                     num_items=None, num_users=None):
# Merge predictions with actual ratings
        merged_df = pd.merge(
            test_data_df[['user_id', 'movie_id', 'rating']],predictions_df,on=['user_id', 'movie_id'], how='inner'
        )
        results = {}
# Rating Prediction Metrics
        results['RMSE'] = calculate_rmse(
            merged_df['predicted_rating'].values,merged_df['rating'].values
        )
        results['MAE'] = calculate_mae(
            merged_df['predicted_rating'].values,merged_df['rating'].values
        )
 # Ranking Metrics (per user, then averaged)
        user_ids = merged_df['user_id'].unique()
        # Initialize metric accumulators
        metrics_per_k = {k: {
            'ndcg': [], 'precision': [], 'recall': [],
            'hit_rate': [], 'mrr': [], 'map': []
        } for k in k_values}
        all_recommended_items = []
        users_with_recs = 0

        for user_id in user_ids:
            user_data = merged_df[merged_df['user_id'] == user_id].copy()
            if len(user_data) == 0:
                continue
            users_with_recs += 1
# Sort by predicted rating (descending) to get ranked recommendations
            user_data = user_data.sort_values('predicted_rating', ascending=False)
            recommended_items = user_data['movie_id'].tolist()
            actual_ratings = user_data['rating'].tolist()
            # Relevant items are those with rating >= threshold
            relevant_items = user_data[
                user_data['rating'] >= self.relevance_threshold
            ]['movie_id'].tolist()
            # Track all recommended items for coverage
            all_recommended_items.extend(recommended_items)

 # Calculate metrics for each K
            for k in k_values:
                # NDCG@K
                relevance_scores = actual_ratings[:k]
                ideal_scores = sorted(actual_ratings, reverse=True)[:k]
                ndcg = ndcg_at_k(relevance_scores, ideal_scores, k)
                metrics_per_k[k]['ndcg'].append(ndcg)
                # Precision@K
                prec = precision_at_k(recommended_items, relevant_items, k)
                metrics_per_k[k]['precision'].append(prec)
                # Recall@K
                rec = recall_at_k(recommended_items, relevant_items, k)
                metrics_per_k[k]['recall'].append(rec)
                # Hit Rate@K
                hr = hit_rate_at_k(recommended_items, relevant_items, k)
                metrics_per_k[k]['hit_rate'].append(hr)
                # MRR (computed on top-K)
                mrr = reciprocal_rank(recommended_items[:k], relevant_items)
                metrics_per_k[k]['mrr'].append(mrr)
                # MAP (computed on top-K)
                ap = average_precision(recommended_items[:k], relevant_items)
                metrics_per_k[k]['map'].append(ap)

# Average metrics across users
        for k in k_values:
            results[f'NDCG@{k}'] = np.mean(metrics_per_k[k]['ndcg'])
            results[f'Precision@{k}'] = np.mean(metrics_per_k[k]['precision'])
            results[f'Recall@{k}'] = np.mean(metrics_per_k[k]['recall'])
            results[f'HitRate@{k}'] = np.mean(metrics_per_k[k]['hit_rate'])
            results[f'MRR@{k}'] = np.mean(metrics_per_k[k]['mrr'])
            results[f'MAP@{k}'] = np.mean(metrics_per_k[k]['map'])

# Coverage Metrics
        if num_items is not None:
            results['CatalogCoverage'] = catalog_coverage(all_recommended_items, num_items)
        if num_users is not None:
            results['UserCoverage'] = user_coverage(users_with_recs, num_users)
        return results

    def print_results(self, results, k_values=[5, 10, 20]):
        print('=' * 60)
        print('RECOMMENDER SYSTEM EVALUATION RESULTS')
        # Rating Prediction Metrics
        print('\nRATING PREDICTION METRICS')
        print(f"  RMSE:{results['RMSE']:.4f}")
        print(f"  MAE: {results['MAE']:.4f}")
        # Ranking Metrics
        print('\n RANKING METRICS')
        # Create a table for ranking metrics
        header = f"{'Metric':<15}" + "".join([f"@{k:<8}" for k in k_values])
        print(header)
        print('-' * len(header))
        for metric in ['NDCG', 'Precision', 'Recall', 'HitRate', 'MRR', 'MAP']:
            row = f"{metric:<15}"
            for k in k_values:
                value = results.get(f'{metric}@{k}', 0)
                row += f"{value:<9.4f}"
            print(row)

        # Coverage Metrics
        if 'CatalogCoverage' in results or 'UserCoverage' in results:
            print('\n COVERAGE METRICS')
            if 'CatalogCoverage' in results:
                print(f"  Catalog Coverage: {results['CatalogCoverage']:.4f} "
                      f"({results['CatalogCoverage']*100:.2f}%)")
            if 'UserCoverage' in results:
                print(f"  User Coverage:    {results['UserCoverage']:.4f} "
                      f"({results['UserCoverage']*100:.2f}%)")
# Convert to DF
predictions_df = pd.DataFrame(
    test_predictions,
    columns=['user_id', 'movie_id', 'predicted_rating']
)
# Prepare test data DataFrame
test_data_df = filtered_test_data[['user_id', 'movie_id', 'rating']].copy()
# Initialize evaluator (items with rating >= 4 are considered relevant)
evaluator = RecommenderMetrics(relevance_threshold=4)
# Compute all metrics
results = evaluator.evaluate_all(
    predictions_df=predictions_df,test_data_df=test_data_df,k_values=[5, 10, 20],num_items=num_movies,num_users=num_users
)
evaluator.print_results(results, k_values=[5, 10, 20])

RECOMMENDER SYSTEM EVALUATION RESULTS

RATING PREDICTION METRICS
  RMSE:5.5583
  MAE: 4.9059

 RANKING METRICS
Metric         @5       @10      @20      
------------------------------------------
NDCG           0.8618   0.8865   0.8930   
Precision      0.5654   0.3476   0.1888   
Recall         0.9075   0.9740   0.9910   
HitRate        0.9941   0.9941   0.9941   
MRR            0.9868   0.9868   0.9868   
MAP            0.9000   0.9646   0.9811   

 COVERAGE METRICS
  Catalog Coverage: 0.6939 (69.39%)
  User Coverage:    0.6990 (69.90%)
